# Enveda CASMI 2026 — Retrieval + Symbolic

v0.3 rdkit-install fix

Internet **off**. Output: `submission.csv`.


In [ ]:
import sys, subprocess
from pathlib import Path

WHEELS = Path('/kaggle/input/rdkit-cp312-wheels-casmi26')
if not WHEELS.is_dir():
    hits = [p for p in Path('/kaggle/input').glob('*/rdkit-*.whl')]
    assert hits, 'rdkit wheels dataset not attached'
    WHEELS = hits[0].parent
print('wheels', sorted(p.name for p in WHEELS.glob('*.whl')))

def pip_offline(*args):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-index',
         '--find-links', str(WHEELS), *args],
        check=True,
    )

try:
    import rdkit  # already present?
except ModuleNotFoundError:
    pip_offline('rdkit')

import rdkit
from rdkit import Chem
print('rdkit', Chem.rdBase.rdkitVersion)


In [ ]:
import sys, os, time, json, random, types
from pathlib import Path
import numpy as np, pandas as pd, pyarrow, pyarrow.parquet as pq
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
print('python', sys.version.split()[0])
print('numpy', np.__version__, 'pandas', pd.__version__, 'pyarrow', pyarrow.__version__)
print('rdkit', Chem.rdBase.rdkitVersion)
SEED=42; random.seed(SEED); np.random.seed(SEED)

DATA=None
for p in [
    Path('/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra'),
    Path('/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'),
]:
    if (p/'test.parquet').exists():
        DATA=p; break
if DATA is None:
    hits=list(Path('/kaggle/input').rglob('test.parquet'))
    assert hits, 'test.parquet not found under /kaggle/input'
    DATA=hits[0].parent
print('DATA', DATA)
OUT=Path('/kaggle/working'); ART=OUT/'artifacts'; ART.mkdir(parents=True, exist_ok=True)
PKG=OUT/'casmi26_pkg'; (PKG/'casmi26').mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PKG))

SOURCES = {
    'adducts': '"""Explicit adduct → neutral-mass conversion rules.\n\nAll supported test-set adducts are unit-tested. Unsupported adducts raise and are logged\nby callers; they are never silently ignored.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\n# IUPAC / CODATA-ish monoisotopic masses\nMASS: Dict[str, float] = {\n    "e": 0.000548579909065,  # electron\n    "H": 1.00782503224,\n    "C": 12.0,\n    "N": 14.00307400443,\n    "O": 15.99491461957,\n    "Na": 22.9897692820,\n    "Cl": 34.968852682,  # 35Cl\n    "K": 38.9637064864,\n}\n\nH_PLUS = MASS["H"] - MASS["e"]  # 1.00727645233\nNA_PLUS = MASS["Na"] - MASS["e"]\nK_PLUS = MASS["K"] - MASS["e"]\nNH4_PLUS = MASS["N"] + 4 * MASS["H"] - MASS["e"]\nCL_MINUS = MASS["Cl"] + MASS["e"]\n# [M+CH2O2-H]- ≡ [M+HCOO]- : adduct ion mass contribution ≈ CHO2 + e for m/z of anion\nCHO2 = MASS["C"] + MASS["H"] + 2 * MASS["O"]  # formyl/formate heavy-atom+H skeleton\nHCOO_MINUS = CHO2 + MASS["e"]\n\n\n@dataclass(frozen=True)\nclass AdductRule:\n    name: str\n    charge: int\n    polarity: str  # "positive" | "negative"\n    # ion_mz = (neutral_mass + delta) / abs(charge)\n    # so neutral_mass = ion_mz * abs(charge) - delta\n    delta: float\n    description: str\n\n    def neutral_mass(self, precursor_mz: float) -> float:\n        if self.charge == 0:\n            raise ValueError(f"Invalid charge for adduct {self.name}")\n        return precursor_mz * abs(self.charge) - self.delta\n\n    def precursor_mz(self, neutral_mass: float) -> float:\n        return (neutral_mass + self.delta) / abs(self.charge)\n\n\n# delta such that: precursor_mz ≈ (M + delta) / |z|\n# [M+H]+ : precursor = M + H+  => delta = +H_PLUS\n# [M-H]- : precursor = M - H+  => delta = -H_PLUS\n# [M+Na]+ : precursor = M + Na+ => delta = +NA_PLUS\n# [M+Cl]- : precursor = M + Cl- => delta = +CL_MINUS\n# [M+CH2O2-H]- : precursor = M + HCOO- => delta = +HCOO_MINUS\nADDUCT_RULES: Dict[str, AdductRule] = {\n    "[M+H]+": AdductRule("[M+H]+", 1, "positive", H_PLUS, "protonated"),\n    "[M-H]-": AdductRule("[M-H]-", -1, "negative", -H_PLUS, "deprotonated"),\n    "[M+Na]+": AdductRule("[M+Na]+", 1, "positive", NA_PLUS, "sodium adduct"),\n    "[M+K]+": AdductRule("[M+K]+", 1, "positive", K_PLUS, "potassium adduct"),\n    "[M+NH4]+": AdductRule("[M+NH4]+", 1, "positive", NH4_PLUS, "ammonium adduct"),\n    "[M+Cl]-": AdductRule("[M+Cl]-", -1, "negative", CL_MINUS, "chloride adduct"),\n    "[M+CH2O2-H]-": AdductRule(\n        "[M+CH2O2-H]-", -1, "negative", HCOO_MINUS, "formate adduct"\n    ),\n    # Extra training adducts (used when indexing train; not required in test)\n    "[M]+": AdductRule("[M]+", 1, "positive", -MASS["e"], "radical cation"),\n    "[2M+H]+": AdductRule("[2M+H]+", 1, "positive", H_PLUS, "dimer protonated"),\n    "[2M-H]-": AdductRule("[2M-H]-", -1, "negative", -H_PLUS, "dimer deprotonated"),\n    "[2M+Na]+": AdductRule("[2M+Na]+", 1, "positive", NA_PLUS, "dimer sodium"),\n    "[2M+CH2O2-H]-": AdductRule(\n        "[2M+CH2O2-H]-", -1, "negative", HCOO_MINUS, "dimer formate"\n    ),\n}\n\nDIMER_ADDUCTS = {"[2M+H]+", "[2M-H]-", "[2M+Na]+", "[2M+CH2O2-H]-"}\n\n\n@dataclass\nclass NeutralMassResult:\n    adduct: str\n    precursor_mz: float\n    neutral_mass: float\n    charge: int\n    polarity: str\n    supported: bool\n    note: str = ""\n    monomer_mass: Optional[float] = None  # for dimers: estimated monomer mass\n\n\ndef parse_adduct(adduct: str) -> AdductRule:\n    if adduct not in ADDUCT_RULES:\n        raise KeyError(f"Unsupported adduct: {adduct!r}")\n    return ADDUCT_RULES[adduct]\n\n\ndef infer_neutral_mass(precursor_mz: float, adduct: str) -> NeutralMassResult:\n    if adduct not in ADDUCT_RULES:\n        return NeutralMassResult(\n            adduct=adduct,\n            precursor_mz=float(precursor_mz),\n            neutral_mass=float("nan"),\n            charge=0,\n            polarity="unknown",\n            supported=False,\n            note="unsupported_adduct",\n        )\n    rule = ADDUCT_RULES[adduct]\n    nm = rule.neutral_mass(float(precursor_mz))\n    monomer = None\n    note = ""\n    if adduct in DIMER_ADDUCTS:\n        # For [2M+X], ion mass ≈ 2M + delta => monomer ≈ nm/2 is wrong because\n        # our delta is applied once. Correct: precursor = 2*M + delta => M = (p - delta)/2\n        monomer = (float(precursor_mz) - rule.delta) / 2.0\n        note = "dimer"\n        nm = monomer  # treat as monomer mass for structure retrieval\n    return NeutralMassResult(\n        adduct=adduct,\n        precursor_mz=float(precursor_mz),\n        neutral_mass=float(nm),\n        charge=rule.charge,\n        polarity=rule.polarity,\n        supported=True,\n        note=note,\n        monomer_mass=monomer,\n    )\n\n\ndef ppm_error(observed: float, expected: float) -> float:\n    if expected == 0 or expected != expected:\n        return float("inf")\n    return (observed - expected) / expected * 1e6\n\n\ndef mass_within_tol(observed: float, expected: float, tol_ppm: float) -> bool:\n    return abs(ppm_error(observed, expected)) <= tol_ppm\n\n\ndef polarity_compatible(adduct: str, ionization_mode: Optional[str]) -> Tuple[bool, str]:\n    if adduct not in ADDUCT_RULES:\n        return False, "unsupported_adduct"\n    rule = ADDUCT_RULES[adduct]\n    if ionization_mode is None or ionization_mode == "" or ionization_mode != ionization_mode:\n        return True, "mode_missing"\n    mode = str(ionization_mode).lower()\n    if mode not in ("positive", "negative"):\n        return True, "mode_unknown"\n    if mode != rule.polarity:\n        return False, "polarity_mismatch"\n    return True, "ok"\n',
    'chemistry': '"""RDKit standardization helpers aligned with competition identity rules."""\n\nfrom __future__ import annotations\n\nfrom functools import lru_cache\nfrom typing import Optional, Tuple\n\nfrom rdkit import Chem\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\nfrom rdkit.Chem import Descriptors\n\n\n_TE = rdMolStandardize.TautomerEnumerator()\n\n\ndef mol_from_smiles(smiles: str) -> Optional[Chem.Mol]:\n    if smiles is None:\n        return None\n    mol = Chem.MolFromSmiles(str(smiles))\n    if mol is None:\n        return None\n    try:\n        Chem.SanitizeMol(mol)\n    except Exception:\n        return None\n    return mol\n\n\ndef tautomer_canonical_smiles(smiles: str) -> Optional[str]:\n    mol = mol_from_smiles(smiles)\n    if mol is None:\n        return None\n    try:\n        canon = _TE.Canonicalize(mol)\n        return Chem.MolToSmiles(canon, isomericSmiles=False)\n    except Exception:\n        try:\n            return Chem.MolToSmiles(mol, isomericSmiles=False)\n        except Exception:\n            return None\n\n\ndef inchikey14_from_smiles(smiles: str, tautomer_canonical: bool = False) -> Optional[str]:\n    mol = mol_from_smiles(smiles)\n    if mol is None:\n        return None\n    try:\n        if tautomer_canonical:\n            mol = _TE.Canonicalize(mol)\n        ik = Chem.MolToInchiKey(mol)\n    except Exception:\n        return None\n    if not ik or len(ik) < 14:\n        return None\n    return ik[:14]\n\n\ndef exact_mass_from_smiles(smiles: str) -> Optional[float]:\n    mol = mol_from_smiles(smiles)\n    if mol is None:\n        return None\n    try:\n        return float(Descriptors.ExactMolWt(mol))\n    except Exception:\n        return None\n\n\n@lru_cache(maxsize=300_000)\ndef standardize_record(smiles: str) -> Tuple[Optional[str], Optional[str], Optional[float]]:\n    """Return (canonical_smiles, inchikey14, exact_mass)."""\n    mol = mol_from_smiles(smiles)\n    if mol is None:\n        return None, None, None\n    try:\n        canon_mol = _TE.Canonicalize(mol)\n    except Exception:\n        canon_mol = mol\n    try:\n        canon_smi = Chem.MolToSmiles(canon_mol, isomericSmiles=False)\n        ik = Chem.MolToInchiKey(canon_mol)\n        mass = float(Descriptors.ExactMolWt(canon_mol))\n    except Exception:\n        return None, None, None\n    if not ik or len(ik) < 14:\n        return None, None, None\n    return canon_smi, ik[:14], mass\n\n\ndef is_valid_smiles(smiles: str) -> bool:\n    return mol_from_smiles(smiles) is not None\n',
    'spectrum': '"""Deterministic spectrum cleaning and similarity."""\n\nfrom __future__ import annotations\n\nfrom typing import Optional, Tuple\n\nimport numpy as np\n\n\ndef clean_spectrum(\n    mzs,\n    intensities,\n    precursor_mz: Optional[float] = None,\n    intensity_floor: float = 0.001,\n    top_peaks: int = 64,\n    merge_tol: float = 0.005,\n    remove_above_precursor_da: float = 2.0,\n    sqrt_intensity: bool = True,\n) -> Tuple[np.ndarray, np.ndarray]:\n    """Return cleaned (mz, intensity) float32 arrays sorted by mz."""\n    if mzs is None or intensities is None:\n        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.float32)\n\n    mz = np.asarray(mzs, dtype=np.float64)\n    inten = np.asarray(intensities, dtype=np.float64)\n    if mz.size == 0 or inten.size == 0 or mz.size != inten.size:\n        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.float32)\n\n    mask = np.isfinite(mz) & np.isfinite(inten) & (mz > 0) & (inten >= 0)\n    if precursor_mz is not None and np.isfinite(precursor_mz):\n        mask &= mz <= (float(precursor_mz) + remove_above_precursor_da)\n    mz = mz[mask]\n    inten = inten[mask]\n    if mz.size == 0:\n        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.float32)\n\n    order = np.argsort(mz)\n    mz = mz[order]\n    inten = inten[order]\n\n    # merge nearby peaks (intensity-weighted mz)\n    merged_mz = []\n    merged_int = []\n    cur_mz = mz[0]\n    cur_int = inten[0]\n    for i in range(1, len(mz)):\n        if mz[i] - cur_mz <= merge_tol:\n            total = cur_int + inten[i]\n            if total > 0:\n                cur_mz = (cur_mz * cur_int + mz[i] * inten[i]) / total\n            cur_int = total\n        else:\n            merged_mz.append(cur_mz)\n            merged_int.append(cur_int)\n            cur_mz = mz[i]\n            cur_int = inten[i]\n    merged_mz.append(cur_mz)\n    merged_int.append(cur_int)\n    mz = np.asarray(merged_mz, dtype=np.float64)\n    inten = np.asarray(merged_int, dtype=np.float64)\n\n    max_i = inten.max()\n    if max_i <= 0:\n        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.float32)\n    inten = inten / max_i\n    keep = inten >= intensity_floor\n    mz = mz[keep]\n    inten = inten[keep]\n    if mz.size == 0:\n        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.float32)\n\n    if mz.size > top_peaks:\n        top_idx = np.argpartition(inten, -top_peaks)[-top_peaks:]\n        top_idx.sort()\n        mz = mz[top_idx]\n        inten = inten[top_idx]\n        # re-sort by mz\n        order = np.argsort(mz)\n        mz = mz[order]\n        inten = inten[order]\n\n    if sqrt_intensity:\n        inten = np.sqrt(inten)\n\n    # L2 normalize for cosine-like scoring\n    norm = np.linalg.norm(inten)\n    if norm > 0:\n        inten = inten / norm\n\n    return mz.astype(np.float32), inten.astype(np.float32)\n\n\ndef _greedy_match(\n    mz_a: np.ndarray,\n    mz_b: np.ndarray,\n    tol: float,\n    shift: float = 0.0,\n) -> Tuple[np.ndarray, np.ndarray]:\n    """Greedy 1-1 matching of peaks within tol after applying shift to b."""\n    i = j = 0\n    pairs_a = []\n    pairs_b = []\n    used_b = np.zeros(len(mz_b), dtype=bool)\n    # two-pointer with local search for closest unused b\n    for i, ma in enumerate(mz_a):\n        target = ma - shift\n        # advance j near target\n        while j < len(mz_b) and mz_b[j] < target - tol:\n            j += 1\n        best_j = -1\n        best_diff = tol + 1\n        k = j\n        while k < len(mz_b) and mz_b[k] <= target + tol:\n            if not used_b[k]:\n                diff = abs(mz_b[k] - target)\n                if diff < best_diff:\n                    best_diff = diff\n                    best_j = k\n            k += 1\n        if best_j >= 0:\n            used_b[best_j] = True\n            pairs_a.append(i)\n            pairs_b.append(best_j)\n    return np.asarray(pairs_a, dtype=np.int64), np.asarray(pairs_b, dtype=np.int64)\n\n\ndef modified_cosine(\n    mz_a: np.ndarray,\n    int_a: np.ndarray,\n    mz_b: np.ndarray,\n    int_b: np.ndarray,\n    precursor_a: float,\n    precursor_b: float,\n    tol: float = 0.05,\n    use_neutral_loss: bool = True,\n) -> float:\n    """Modified cosine with optional precursor-shift (neutral-loss) matching."""\n    if len(mz_a) == 0 or len(mz_b) == 0:\n        return 0.0\n\n    ia, ib = _greedy_match(mz_a, mz_b, tol, shift=0.0)\n    matched = set(zip(ia.tolist(), ib.tolist())) if len(ia) else set()\n\n    if use_neutral_loss and np.isfinite(precursor_a) and np.isfinite(precursor_b):\n        shift = float(precursor_a) - float(precursor_b)\n        if abs(shift) > 1e-6:\n            # rematch unused peaks under shift\n            used_a = {p[0] for p in matched}\n            used_b = {p[1] for p in matched}\n            mz_a2 = mz_a\n            mz_b2 = mz_b\n            ia2, ib2 = _greedy_match(mz_a2, mz_b2, tol, shift=shift)\n            for a_idx, b_idx in zip(ia2.tolist(), ib2.tolist()):\n                if a_idx in used_a or b_idx in used_b:\n                    continue\n                matched.add((a_idx, b_idx))\n                used_a.add(a_idx)\n                used_b.add(b_idx)\n\n    if not matched:\n        return 0.0\n\n    score = 0.0\n    for a_idx, b_idx in matched:\n        score += float(int_a[a_idx]) * float(int_b[b_idx])\n    # already L2-normalized => score in [0, 1]\n    return float(max(0.0, min(1.0, score)))\n\n\ndef _spectral_entropy(intensities: np.ndarray) -> float:\n    s = float(np.sum(intensities))\n    if s <= 0:\n        return 0.0\n    p = intensities / s\n    p = p[p > 0]\n    return float(-np.sum(p * np.log(p)))\n\n\ndef _entropy_weights(intensities: np.ndarray) -> np.ndarray:\n    """Li & Fiehn intensity reweighting based on spectral entropy."""\n    ent = _spectral_entropy(intensities)\n    w = 0.25 + 0.25 * ent  # in [0.25, ~1+]\n    weighted = np.power(np.maximum(intensities, 0.0), w)\n    total = float(np.sum(weighted))\n    if total <= 0:\n        return intensities\n    return weighted / total\n\n\ndef entropy_similarity(\n    mz_a: np.ndarray,\n    int_a: np.ndarray,\n    mz_b: np.ndarray,\n    int_b: np.ndarray,\n    tol: float = 0.05,\n) -> float:\n    """Entropy similarity in [0, 1] after un-doing L2 norm assumption.\n\n    Inputs may be L2-normalized cleaned peaks; we convert to positive weights,\n    apply entropy weighting, merge matched peaks, and score:\n    1 - (2*S_ab - S_a - S_b) / ln(4).\n    """\n    if len(mz_a) == 0 or len(mz_b) == 0:\n        return 0.0\n    # Recover relative intensities (already non-negative)\n    wa = _entropy_weights(np.asarray(int_a, dtype=np.float64) ** 2)  # undo sqrt+L2 approx\n    wb = _entropy_weights(np.asarray(int_b, dtype=np.float64) ** 2)\n    # If ints were not sqrt-scaled L2, fallback to abs values\n    if not np.isfinite(wa).all() or float(np.sum(wa)) <= 0:\n        wa = _entropy_weights(np.abs(np.asarray(int_a, dtype=np.float64)))\n    if not np.isfinite(wb).all() or float(np.sum(wb)) <= 0:\n        wb = _entropy_weights(np.abs(np.asarray(int_b, dtype=np.float64)))\n\n    ia, ib = _greedy_match(mz_a, mz_b, tol, shift=0.0)\n    merged = []\n    used_a = set(ia.tolist()) if len(ia) else set()\n    used_b = set(ib.tolist()) if len(ib) else set()\n    for a_idx, b_idx in zip(ia.tolist(), ib.tolist()) if len(ia) else []:\n        merged.append(wa[a_idx] + wb[b_idx])\n    for i, v in enumerate(wa):\n        if i not in used_a:\n            merged.append(v)\n    for j, v in enumerate(wb):\n        if j not in used_b:\n            merged.append(v)\n    merged = np.asarray(merged, dtype=np.float64)\n    s_a = _spectral_entropy(wa)\n    s_b = _spectral_entropy(wb)\n    s_ab = _spectral_entropy(merged)\n    denom = np.log(4.0)\n    sim = 1.0 - (2.0 * s_ab - s_a - s_b) / denom\n    return float(max(0.0, min(1.0, sim)))\n\n\ndef hybrid_similarity(\n    mz_a: np.ndarray,\n    int_a: np.ndarray,\n    mz_b: np.ndarray,\n    int_b: np.ndarray,\n    precursor_a: float,\n    precursor_b: float,\n    tol: float = 0.05,\n    use_neutral_loss: bool = True,\n    entropy_weight: float = 0.55,\n) -> float:\n    cos = modified_cosine(\n        mz_a, int_a, mz_b, int_b, precursor_a, precursor_b, tol, use_neutral_loss\n    )\n    ent = entropy_similarity(mz_a, int_a, mz_b, int_b, tol=tol)\n    w = float(entropy_weight)\n    return float(w * ent + (1.0 - w) * cos)\n\n\ndef spectra_identical(\n    mz_a: np.ndarray,\n    int_a: np.ndarray,\n    mz_b: np.ndarray,\n    int_b: np.ndarray,\n    precursor_a: float,\n    precursor_b: float,\n    prec_atol: float = 1e-4,\n) -> bool:\n    """True if cleaned spectra + precursor match (poisoned-label guard)."""\n    if abs(float(precursor_a) - float(precursor_b)) > prec_atol:\n        return False\n    if len(mz_a) != len(mz_b) or len(mz_a) == 0:\n        return False\n    return bool(np.array_equal(mz_a, mz_b) and np.array_equal(int_a, int_b))\n',
    'config': '"""Global configuration for the first retrieval submission."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, List\n\nROOT = Path(__file__).resolve().parents[1]\nDATA_DIR = ROOT / "data"\nOUTPUT_DIR = ROOT / "outputs"\nARTIFACT_DIR = ROOT / "artifacts"\n\nSEED = 42\nMAX_CANDIDATES = 25\nMASS_TOL_PPM = 25.0\nMASS_TOL_PPM_BACKFILL = 60.0\nPEAK_MZ_TOL = 0.05\nTOP_PEAKS = 128\nINTENSITY_FLOOR = 0.001\nMAX_PEAKS_ABOVE_PRECURSOR_DA = 2.0\nTOP_K_SPECTRA_PER_QUERY = 80\nTOP_CANDIDATES_PER_SPECTRUM = 50\nMIN_SIMILARITY = 0.08\nENTROPY_WEIGHT = 0.55\nEXCLUDE_EXACT_DUPLICATES = True\n\n# Aggregation weights (normalized features). Tuned for MRR@25 orientation.\nWEIGHTS: Dict[str, float] = {\n    "best_similarity": 1.00,\n    "mean_top_k_similarity": 0.35,\n    "support_log": 0.25,\n    "ce_coverage": 0.10,\n    "adduct_coverage": 0.10,\n    "domain_prior": 0.20,\n    "quality": 0.10,\n    "mass_error": 0.30,\n    "contradiction": 0.50,\n}\n\nDOMAIN_PRIOR: Dict[str, float] = {\n    "enveda-np-examples": 1.00,\n    "enveda-180": 0.95,\n    "gnps": 0.72,\n    "riken": 0.72,\n    "massbank": 0.65,\n    "mona": 0.65,\n    "msdial": 0.60,\n    "spectraverse": 0.60,\n    "pluskal_ms2": 0.45,\n    "drug_plus": 0.40,\n    "masaryk": 0.40,\n}\n\nPREFERRED_LIBS: List[str] = [\n    "enveda-np-examples",\n    "enveda-180",\n    "gnps",\n    "riken",\n    "massbank",\n    "mona",\n    "msdial",\n    "spectraverse",\n]\n\n\n@dataclass\nclass RunConfig:\n    mass_tol_ppm: float = MASS_TOL_PPM\n    mass_tol_ppm_backfill: float = MASS_TOL_PPM_BACKFILL\n    peak_mz_tol: float = PEAK_MZ_TOL\n    top_peaks: int = TOP_PEAKS\n    intensity_floor: float = INTENSITY_FLOOR\n    top_k_spectra: int = TOP_K_SPECTRA_PER_QUERY\n    top_candidates_per_spectrum: int = TOP_CANDIDATES_PER_SPECTRUM\n    min_similarity: float = MIN_SIMILARITY\n    max_candidates: int = MAX_CANDIDATES\n    seed: int = SEED\n    use_neutral_loss: bool = True\n    prefer_same_mode: bool = True\n    entropy_weight: float = ENTROPY_WEIGHT\n    exclude_exact_duplicates: bool = EXCLUDE_EXACT_DUPLICATES\n    weights: Dict[str, float] = field(default_factory=lambda: dict(WEIGHTS))\n    domain_prior: Dict[str, float] = field(default_factory=lambda: dict(DOMAIN_PRIOR))\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n',
    'index': '"""Mass-pruned spectral retrieval index."""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\nfrom tqdm import tqdm\n\nfrom .adducts import ADDUCT_RULES, DIMER_ADDUCTS, infer_neutral_mass\nfrom .config import ARTIFACT_DIR, DOMAIN_PRIOR, RunConfig\nfrom .spectrum import clean_spectrum\n\n\n@dataclass\nclass SpectrumIndex:\n    neutral_mass: np.ndarray\n    precursor_mz: np.ndarray\n    order: np.ndarray\n    inchikey14: np.ndarray\n    smiles: np.ndarray\n    library: np.ndarray\n    adduct: np.ndarray\n    ionization_mode: np.ndarray\n    quality: np.ndarray\n    exact_mass: np.ndarray\n    domain_prior: np.ndarray\n    peak_mz: np.ndarray\n    peak_intensity: np.ndarray\n    collision_energy: np.ndarray\n\n    def save(self, path: Path) -> None:\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        np.savez_compressed(\n            path,\n            neutral_mass=self.neutral_mass,\n            precursor_mz=self.precursor_mz,\n            order=self.order,\n            inchikey14=self.inchikey14,\n            smiles=self.smiles,\n            library=self.library,\n            adduct=self.adduct,\n            ionization_mode=self.ionization_mode,\n            quality=self.quality,\n            exact_mass=self.exact_mass,\n            domain_prior=self.domain_prior,\n            peak_mz=self.peak_mz,\n            peak_intensity=self.peak_intensity,\n            collision_energy=self.collision_energy,\n        )\n\n    @classmethod\n    def load(cls, path: Path) -> "SpectrumIndex":\n        data = np.load(path, allow_pickle=True)\n        return cls(**{k: data[k] for k in data.files})\n\n    def mass_window(self, mass: float, tol_ppm: float) -> Tuple[int, int]:\n        if not np.isfinite(mass) or mass <= 0:\n            return 0, 0\n        delta = mass * tol_ppm * 1e-6\n        lo = int(np.searchsorted(self.neutral_mass, mass - delta, side="left"))\n        hi = int(np.searchsorted(self.neutral_mass, mass + delta, side="right"))\n        return lo, hi\n\n\ndef _mean_ce(val) -> float:\n    if val is None:\n        return float("nan")\n    arr = np.asarray(val, dtype=np.float64)\n    if arr.size == 0:\n        return float("nan")\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(arr.mean())\n\n\ndef _quality_score(precursor_error_ppm, num_peaks) -> float:\n    err = abs(float(precursor_error_ppm)) if precursor_error_ppm == precursor_error_ppm else 1e6\n    q_err = 1.0 / (1.0 + err / 5.0)\n    peaks = float(num_peaks) if num_peaks == num_peaks else 0.0\n    q_peaks = min(1.0, peaks / 64.0)\n    return 0.7 * q_err + 0.3 * q_peaks\n\n\ndef _vector_neutral_mass(precursor_mz: np.ndarray, adducts: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    """Fast adduct → neutral mass for arrays. Returns (neutral_mass, supported_mask)."""\n    n = len(precursor_mz)\n    out = np.full(n, np.nan, dtype=np.float64)\n    supported = np.zeros(n, dtype=bool)\n    # group by adduct string\n    adducts = np.asarray(adducts, dtype=object)\n    for adduct, rule in ADDUCT_RULES.items():\n        mask = adducts == adduct\n        if not np.any(mask):\n            continue\n        p = precursor_mz[mask].astype(np.float64)\n        if adduct in DIMER_ADDUCTS:\n            nm = (p - rule.delta) / 2.0\n        else:\n            nm = p * abs(rule.charge) - rule.delta\n        out[mask] = nm\n        supported[mask] = True\n    return out, supported\n\n\ndef build_index_from_train(\n    train_path: Path,\n    cfg: RunConfig,\n    cache_path: Optional[Path] = None,\n    max_rows: Optional[int] = None,\n    libraries: Optional[Sequence[str]] = None,\n) -> SpectrumIndex:\n    cache_path = Path(cache_path) if cache_path else ARTIFACT_DIR / "spectrum_index.npz"\n    meta_path = cache_path.with_suffix(".meta.json")\n    if cache_path.exists():\n        print(f"[index] loading cache {cache_path}")\n        return SpectrumIndex.load(cache_path)\n\n    t0 = time.time()\n    pf = pq.ParquetFile(train_path)\n    cols = [\n        "ingest_lib",\n        "normalized_smiles",\n        "inchikey14",\n        "ionization_mode",\n        "adduct",\n        "precursor_mz",\n        "precursor_error_ppm",\n        "ms2_mzs",\n        "ms2_normalized_intensities",\n        "num_peaks",\n        "collision_energy_ev",\n    ]\n\n    chunks = []\n    n_unsupported = 0\n    n_bad_peaks = 0\n    n_kept = 0\n    lib_filter = set(libraries) if libraries else None\n\n    for rg_idx in tqdm(range(pf.num_row_groups), desc="index row_groups"):\n        if max_rows is not None and n_kept >= max_rows:\n            break\n        df = pf.read_row_group(rg_idx, columns=cols).to_pandas()\n        if lib_filter is not None:\n            df = df[df["ingest_lib"].isin(lib_filter)]\n        if df.empty:\n            continue\n        if max_rows is not None:\n            remain = max_rows - n_kept\n            if len(df) > remain:\n                df = df.iloc[:remain]\n\n        precursor = df["precursor_mz"].to_numpy(dtype=np.float64)\n        neutral, ok = _vector_neutral_mass(precursor, df["adduct"].to_numpy())\n        n_unsupported += int((~ok).sum())\n        df = df.loc[ok].copy()\n        neutral = neutral[ok]\n        precursor = precursor[ok]\n        if df.empty:\n            continue\n\n        peak_mz = []\n        peak_int = []\n        keep_mask = []\n        mzs_col = df["ms2_mzs"].to_numpy()\n        ints_col = df["ms2_normalized_intensities"].to_numpy()\n        for i in range(len(df)):\n            mz, inten = clean_spectrum(\n                mzs_col[i],\n                ints_col[i],\n                precursor_mz=float(precursor[i]),\n                intensity_floor=cfg.intensity_floor,\n                top_peaks=cfg.top_peaks,\n            )\n            if len(mz) < 3:\n                keep_mask.append(False)\n                peak_mz.append(None)\n                peak_int.append(None)\n            else:\n                keep_mask.append(True)\n                peak_mz.append(mz)\n                peak_int.append(inten)\n        keep_mask = np.asarray(keep_mask, dtype=bool)\n        n_bad_peaks += int((~keep_mask).sum())\n        if not keep_mask.any():\n            continue\n\n        df = df.loc[keep_mask]\n        neutral = neutral[keep_mask]\n        precursor = precursor[keep_mask]\n        peak_mz = [peak_mz[i] for i, k in enumerate(keep_mask) if k]\n        peak_int = [peak_int[i] for i, k in enumerate(keep_mask) if k]\n\n        libs = df["ingest_lib"].fillna("unknown").astype(str)\n        err = df["precursor_error_ppm"].to_numpy()\n        npeaks = df["num_peaks"].to_numpy()\n        quality = np.array([_quality_score(e, p) for e, p in zip(err, npeaks)], dtype=np.float32)\n        prior = libs.map(lambda x: float(DOMAIN_PRIOR.get(x, 0.3))).to_numpy(dtype=np.float32)\n        ce = np.array([_mean_ce(v) for v in df["collision_energy_ev"].to_numpy()], dtype=np.float32)\n\n        chunk = {\n            "neutral_mass": neutral.astype(np.float64),\n            "precursor_mz": precursor.astype(np.float64),\n            "inchikey14": df["inchikey14"].to_numpy(dtype=object),\n            "smiles": df["normalized_smiles"].to_numpy(dtype=object),\n            "library": libs.to_numpy(dtype=object),\n            "adduct": df["adduct"].to_numpy(dtype=object),\n            "ionization_mode": df["ionization_mode"].to_numpy(dtype=object),\n            "quality": quality,\n            "exact_mass": neutral.astype(np.float64),\n            "domain_prior": prior,\n            "peak_mz": np.asarray(peak_mz, dtype=object),\n            "peak_intensity": np.asarray(peak_int, dtype=object),\n            "collision_energy": ce,\n        }\n        chunks.append(chunk)\n        n_kept += len(df)\n\n    if not chunks:\n        raise RuntimeError("No spectra retained while building index")\n\n    def cat(key, dtype=None):\n        arrs = [c[key] for c in chunks]\n        if dtype is object or arrs[0].dtype == object:\n            return np.concatenate(arrs)\n        return np.concatenate(arrs).astype(dtype) if dtype else np.concatenate(arrs)\n\n    neutral_mass = cat("neutral_mass", np.float64)\n    order = np.argsort(neutral_mass)\n    index = SpectrumIndex(\n        neutral_mass=neutral_mass[order],\n        precursor_mz=cat("precursor_mz", np.float64)[order],\n        order=order,\n        inchikey14=cat("inchikey14")[order],\n        smiles=cat("smiles")[order],\n        library=cat("library")[order],\n        adduct=cat("adduct")[order],\n        ionization_mode=cat("ionization_mode")[order],\n        quality=cat("quality", np.float32)[order],\n        exact_mass=cat("exact_mass", np.float64)[order],\n        domain_prior=cat("domain_prior", np.float32)[order],\n        peak_mz=cat("peak_mz")[order],\n        peak_intensity=cat("peak_intensity")[order],\n        collision_energy=cat("collision_energy", np.float32)[order],\n    )\n    index.save(cache_path)\n    meta = {\n        "n_kept": int(n_kept),\n        "n_unsupported": int(n_unsupported),\n        "n_bad_peaks": int(n_bad_peaks),\n        "elapsed_sec": time.time() - t0,\n        "config": cfg.to_dict(),\n    }\n    meta_path.write_text(json.dumps(meta, indent=2))\n    print(\n        f"[index] kept={n_kept} unsupported={n_unsupported} bad_peaks={n_bad_peaks} "\n        f"elapsed={meta[\'elapsed_sec\']:.1f}s -> {cache_path}"\n    )\n    return index\n',
    'retrieve': '"""Per-spectrum retrieval and molecule-level aggregation."""\n\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nfrom .adducts import infer_neutral_mass, mass_within_tol, ppm_error, polarity_compatible\nfrom .chemistry import is_valid_smiles\nfrom .config import RunConfig\nfrom .index import SpectrumIndex\nfrom .spectrum import clean_spectrum, hybrid_similarity, spectra_identical\n\n\n@dataclass\nclass CandidateEvidence:\n    inchikey14: str\n    smiles: str\n    final_score: float = 0.0\n    best_similarity: float = 0.0\n    mean_top_k_similarity: float = 0.0\n    n_supporting_spectra: int = 0\n    n_supporting_query_spectra: int = 0\n    collision_energies: List[float] = field(default_factory=list)\n    adducts: List[str] = field(default_factory=list)\n    libraries: List[str] = field(default_factory=list)\n    mass_error_ppm: float = float("inf")\n    domain_prior: float = 0.0\n    quality: float = 0.0\n    contradictions: List[str] = field(default_factory=list)\n    matched_train_indices: List[int] = field(default_factory=list)\n    query_spectrum_ids: List[str] = field(default_factory=list)\n    backfilled: bool = False\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            "inchikey14": self.inchikey14,\n            "smiles": self.smiles,\n            "final_score": self.final_score,\n            "best_similarity": self.best_similarity,\n            "mean_top_k_similarity": self.mean_top_k_similarity,\n            "n_supporting_spectra": self.n_supporting_spectra,\n            "n_supporting_query_spectra": self.n_supporting_query_spectra,\n            "collision_energies": self.collision_energies,\n            "adducts": self.adducts,\n            "libraries": self.libraries,\n            "mass_error_ppm": self.mass_error_ppm,\n            "domain_prior": self.domain_prior,\n            "quality": self.quality,\n            "contradictions": self.contradictions,\n            "matched_train_indices": self.matched_train_indices,\n            "query_spectrum_ids": self.query_spectrum_ids,\n            "backfilled": self.backfilled,\n        }\n\n\ndef _query_neutral_masses(spectra_df: pd.DataFrame) -> Tuple[float, float, List[dict]]:\n    """Robust combined neutral mass for a molecule from its spectra."""\n    estimates = []\n    details = []\n    for row in spectra_df.itertuples(index=False):\n        res = infer_neutral_mass(row.precursor_mz, row.adduct)\n        ok, note = polarity_compatible(row.adduct, row.ionization_mode)\n        details.append(\n            {\n                "spectrum_id": getattr(row, "spectrum_id", None),\n                "adduct": row.adduct,\n                "precursor_mz": float(row.precursor_mz),\n                "neutral_mass": res.neutral_mass,\n                "supported": res.supported,\n                "polarity_ok": ok,\n                "note": note if res.supported else res.note,\n            }\n        )\n        if res.supported and ok and np.isfinite(res.neutral_mass):\n            # weight by base peak intensity if available\n            w = float(getattr(row, "base_peak_intensity", 1.0) or 1.0)\n            w = max(w, 1.0)\n            estimates.append((res.neutral_mass, w))\n    if not estimates:\n        return float("nan"), float("inf"), details\n    masses = np.array([m for m, _ in estimates], dtype=np.float64)\n    weights = np.array([w for _, w in estimates], dtype=np.float64)\n    # weighted median approximation: weighted mean of central values\n    order = np.argsort(masses)\n    masses = masses[order]\n    weights = weights[order]\n    cum = np.cumsum(weights)\n    cutoff = 0.5 * cum[-1]\n    idx = int(np.searchsorted(cum, cutoff))\n    idx = min(idx, len(masses) - 1)\n    center = float(masses[idx])\n    spread = float(np.ptp(masses)) if len(masses) > 1 else 0.0\n    return center, spread, details\n\n\ndef retrieve_for_spectrum(\n    index: SpectrumIndex,\n    mz: np.ndarray,\n    inten: np.ndarray,\n    precursor_mz: float,\n    neutral_mass: float,\n    ionization_mode: str,\n    adduct: str,\n    cfg: RunConfig,\n    tol_ppm: Optional[float] = None,\n    banned_inchikey14: Optional[set] = None,\n) -> List[Tuple[int, float]]:\n    """Return list of (index_pos, similarity) sorted descending."""\n    tol = cfg.mass_tol_ppm if tol_ppm is None else tol_ppm\n    lo, hi = index.mass_window(neutral_mass, tol)\n    if hi <= lo:\n        return []\n    banned = banned_inchikey14 or set()\n\n    scores: List[Tuple[int, float]] = []\n    for pos in range(lo, hi):\n        ik = index.inchikey14[pos]\n        if ik in banned:\n            continue\n        if cfg.prefer_same_mode and ionization_mode in ("positive", "negative"):\n            mode = index.ionization_mode[pos]\n            if mode in ("positive", "negative") and mode != ionization_mode:\n                continue\n        train_pre = float(index.precursor_mz[pos])\n        # Skip exact duplicates — train labels on leaked identical rows ≠ competition GT.\n        if cfg.exclude_exact_duplicates and spectra_identical(\n            mz, inten, index.peak_mz[pos], index.peak_intensity[pos], precursor_mz, train_pre\n        ):\n            continue\n        sim = hybrid_similarity(\n            mz,\n            inten,\n            index.peak_mz[pos],\n            index.peak_intensity[pos],\n            precursor_a=precursor_mz,\n            precursor_b=train_pre,\n            tol=cfg.peak_mz_tol,\n            use_neutral_loss=cfg.use_neutral_loss,\n            entropy_weight=cfg.entropy_weight,\n        )\n        if sim >= cfg.min_similarity:\n            if index.adduct[pos] == adduct:\n                sim = min(1.0, sim + 0.02)\n            scores.append((pos, sim))\n\n    scores.sort(key=lambda x: x[1], reverse=True)\n    return scores[: cfg.top_k_spectra]\n\n\ndef _poisoned_inchikeys_for_molecule(\n    spectra_df: pd.DataFrame,\n    index: SpectrumIndex,\n    cfg: RunConfig,\n) -> set:\n    """Inchikey14s attached to exact test↔train spectrum duplicates (untrustworthy labels)."""\n    if not cfg.exclude_exact_duplicates:\n        return set()\n    banned = set()\n    for row in spectra_df.itertuples(index=False):\n        nm = infer_neutral_mass(row.precursor_mz, row.adduct)\n        if not nm.supported or not np.isfinite(nm.neutral_mass):\n            continue\n        mz, inten = clean_spectrum(\n            row.ms2_mzs,\n            row.ms2_normalized_intensities,\n            precursor_mz=row.precursor_mz,\n            intensity_floor=cfg.intensity_floor,\n            top_peaks=cfg.top_peaks,\n        )\n        if len(mz) < 2:\n            continue\n        lo, hi = index.mass_window(nm.neutral_mass, cfg.mass_tol_ppm_backfill)\n        pre = float(row.precursor_mz)\n        for pos in range(lo, hi):\n            if spectra_identical(\n                mz, inten, index.peak_mz[pos], index.peak_intensity[pos], pre, float(index.precursor_mz[pos])\n            ):\n                banned.add(index.inchikey14[pos])\n    return banned\n\n\ndef aggregate_molecule_candidates(\n    molecule_id: str,\n    spectra_df: pd.DataFrame,\n    index: SpectrumIndex,\n    cfg: RunConfig,\n) -> Tuple[List[CandidateEvidence], Dict[str, Any]]:\n    """Retrieve + aggregate candidates for one molecule_id."""\n    neutral_mass, mass_spread, mass_details = _query_neutral_masses(spectra_df)\n    diagnostics = {\n        "molecule_id": molecule_id,\n        "n_spectra": len(spectra_df),\n        "neutral_mass": neutral_mass,\n        "mass_spread_da": mass_spread,\n        "mass_details": mass_details,\n        "fallback_used": False,\n        "placeholder_used": False,\n    }\n\n    # per-candidate accumulators\n    acc: Dict[str, dict] = {}\n    banned_iks = _poisoned_inchikeys_for_molecule(spectra_df, index, cfg)\n    diagnostics["banned_poisoned_inchikey14"] = sorted(banned_iks)\n    diagnostics["n_banned_poisoned"] = len(banned_iks)\n\n    for row in spectra_df.itertuples(index=False):\n        nm = infer_neutral_mass(row.precursor_mz, row.adduct)\n        if not nm.supported or not np.isfinite(nm.neutral_mass):\n            continue\n        ok, _ = polarity_compatible(row.adduct, row.ionization_mode)\n        if not ok:\n            continue\n        mz, inten = clean_spectrum(\n            row.ms2_mzs,\n            row.ms2_normalized_intensities,\n            precursor_mz=row.precursor_mz,\n            intensity_floor=cfg.intensity_floor,\n            top_peaks=cfg.top_peaks,\n        )\n        if len(mz) < 2:\n            continue\n        hits = retrieve_for_spectrum(\n            index,\n            mz,\n            inten,\n            precursor_mz=float(row.precursor_mz),\n            neutral_mass=nm.neutral_mass,\n            ionization_mode=str(row.ionization_mode),\n            adduct=str(row.adduct),\n            cfg=cfg,\n            banned_inchikey14=banned_iks,\n        )\n        # map to unique structures, keep best sim per structure for this query spectrum\n        best_for_struct: Dict[str, Tuple[int, float]] = {}\n        for pos, sim in hits:\n            ik = index.inchikey14[pos]\n            prev = best_for_struct.get(ik)\n            if prev is None or sim > prev[1]:\n                best_for_struct[ik] = (pos, sim)\n\n        # retain top structures for this spectrum\n        ranked = sorted(best_for_struct.items(), key=lambda x: x[1][1], reverse=True)\n        ranked = ranked[: cfg.top_candidates_per_spectrum]\n        sid = str(getattr(row, "spectrum_id", ""))\n        ce = row.collision_energy_ev\n        ce_vals = []\n        if ce is not None:\n            arr = np.asarray(ce, dtype=np.float64)\n            ce_vals = [float(x) for x in arr[np.isfinite(arr)]]\n\n        for ik, (pos, sim) in ranked:\n            slot = acc.get(ik)\n            if slot is None:\n                slot = {\n                    "smiles": index.smiles[pos],\n                    "sims": [],\n                    "query_ids": set(),\n                    "train_pos": [],\n                    "adducts": set(),\n                    "ces": set(),\n                    "libs": set(),\n                    "qualities": [],\n                    "priors": [],\n                    "mass_errors": [],\n                    "contradictions": [],\n                }\n                acc[ik] = slot\n            slot["sims"].append(sim)\n            slot["query_ids"].add(sid)\n            slot["train_pos"].append(int(pos))\n            slot["adducts"].add(str(row.adduct))\n            for c in ce_vals:\n                slot["ces"].add(round(c, 1))\n            slot["libs"].add(str(index.library[pos]))\n            slot["qualities"].append(float(index.quality[pos]))\n            slot["priors"].append(float(index.domain_prior[pos]))\n            # mass error vs query inferred neutral mass\n            err = abs(ppm_error(float(index.exact_mass[pos]), nm.neutral_mass))\n            slot["mass_errors"].append(err)\n            if not mass_within_tol(float(index.exact_mass[pos]), nm.neutral_mass, cfg.mass_tol_ppm):\n                slot["contradictions"].append("mass_tol_exceeded")\n\n    candidates: List[CandidateEvidence] = []\n    w = cfg.weights\n    for ik, slot in acc.items():\n        smiles = slot["smiles"]\n        # Hard constraints: trust train inchikey14; only require parseable SMILES.\n        # Full tautomer re-canonicalization is deferred to submission validation.\n        if smiles is None or not is_valid_smiles(smiles):\n            continue\n        if not ik or not isinstance(ik, str) or len(ik) < 14:\n            continue\n        ik = ik[:14]\n\n        sims = sorted(slot["sims"], reverse=True)\n        best_sim = sims[0]\n        mean_top = float(np.mean(sims[: min(5, len(sims))]))\n        n_support = len(sims)\n        n_query = len(slot["query_ids"])\n        ce_cov = min(1.0, len(slot["ces"]) / 3.0)\n        adduct_cov = min(1.0, len(slot["adducts"]) / max(1, spectra_df["adduct"].nunique()))\n        domain = float(np.max(slot["priors"])) if slot["priors"] else 0.0\n        quality = float(np.mean(slot["qualities"])) if slot["qualities"] else 0.0\n        mass_err = float(np.min(slot["mass_errors"])) if slot["mass_errors"] else 1e6\n        mass_err_norm = min(1.0, mass_err / max(cfg.mass_tol_ppm, 1.0))\n        contradictions = sorted(set(slot["contradictions"]))\n        contrad_pen = 1.0 if contradictions else 0.0\n\n        # reject hard mass contradictions beyond backfill tolerance\n        if mass_err > cfg.mass_tol_ppm_backfill:\n            continue\n\n        score = (\n            w["best_similarity"] * best_sim\n            + w["mean_top_k_similarity"] * mean_top\n            + w["support_log"] * (np.log1p(n_support) / np.log1p(20))\n            + w["ce_coverage"] * ce_cov\n            + w["adduct_coverage"] * adduct_cov\n            + w["domain_prior"] * domain\n            + w["quality"] * quality\n            - w["mass_error"] * mass_err_norm\n            - w["contradiction"] * contrad_pen\n        )\n        candidates.append(\n            CandidateEvidence(\n                inchikey14=ik,\n                smiles=smiles,\n                final_score=float(score),\n                best_similarity=float(best_sim),\n                mean_top_k_similarity=float(mean_top),\n                n_supporting_spectra=n_support,\n                n_supporting_query_spectra=n_query,\n                collision_energies=sorted(slot["ces"]),\n                adducts=sorted(slot["adducts"]),\n                libraries=sorted(slot["libs"]),\n                mass_error_ppm=mass_err,\n                domain_prior=domain,\n                quality=quality,\n                contradictions=contradictions,\n                matched_train_indices=slot["train_pos"][:20],\n                query_spectrum_ids=sorted(slot["query_ids"]),\n            )\n        )\n\n    # dedupe by inchikey14 keeping best score\n    by_ik: Dict[str, CandidateEvidence] = {}\n    for c in candidates:\n        prev = by_ik.get(c.inchikey14)\n        if prev is None or c.final_score > prev.final_score:\n            by_ik[c.inchikey14] = c\n    candidates = sorted(by_ik.values(), key=lambda c: c.final_score, reverse=True)\n\n    # backfill if fewer than 25 using wider mass window on best query spectrum\n    if len(candidates) < cfg.max_candidates and np.isfinite(neutral_mass):\n        diagnostics["fallback_used"] = True\n        have = {c.inchikey14 for c in candidates}\n        # use spectrum with most peaks\n        best_row = max(\n            spectra_df.itertuples(index=False),\n            key=lambda r: len(r.ms2_mzs) if r.ms2_mzs is not None else 0,\n        )\n        nm = infer_neutral_mass(best_row.precursor_mz, best_row.adduct)\n        mz, inten = clean_spectrum(\n            best_row.ms2_mzs,\n            best_row.ms2_normalized_intensities,\n            precursor_mz=best_row.precursor_mz,\n            intensity_floor=cfg.intensity_floor,\n            top_peaks=cfg.top_peaks,\n        )\n        hits = retrieve_for_spectrum(\n            index,\n            mz,\n            inten,\n            precursor_mz=float(best_row.precursor_mz),\n            neutral_mass=nm.neutral_mass if nm.supported else neutral_mass,\n            ionization_mode=str(best_row.ionization_mode),\n            adduct=str(best_row.adduct),\n            cfg=cfg,\n            tol_ppm=cfg.mass_tol_ppm_backfill,\n            banned_inchikey14=banned_iks,\n        )\n        for pos, sim in hits:\n            ik = index.inchikey14[pos]\n            if ik in have or ik in banned_iks:\n                continue\n            smiles = index.smiles[pos]\n            if smiles is None or not is_valid_smiles(smiles):\n                continue\n            ik2 = str(ik)[:14]\n            if ik2 in have or ik2 in banned_iks:\n                continue\n            candidates.append(\n                CandidateEvidence(\n                    inchikey14=ik2,\n                    smiles=smiles,\n                    final_score=float(sim) * 0.1,  # clearly lower than primary\n                    best_similarity=float(sim),\n                    mean_top_k_similarity=float(sim),\n                    n_supporting_spectra=1,\n                    n_supporting_query_spectra=1,\n                    libraries=[str(index.library[pos])],\n                    mass_error_ppm=abs(ppm_error(float(index.exact_mass[pos]), neutral_mass)),\n                    domain_prior=float(index.domain_prior[pos]),\n                    quality=float(index.quality[pos]),\n                    backfilled=True,\n                    query_spectrum_ids=[str(getattr(best_row, "spectrum_id", ""))],\n                    matched_train_indices=[int(pos)],\n                )\n            )\n            have.add(ik2)\n            if len(candidates) >= cfg.max_candidates:\n                break\n        candidates = sorted(candidates, key=lambda c: c.final_score, reverse=True)\n\n    # ultimate placeholder to keep submission valid\n    if not candidates:\n        diagnostics["placeholder_used"] = True\n        candidates = [\n            CandidateEvidence(\n                inchikey14="LFQSCWFLJHTTHZ",  # ethanol\n                smiles="CCO",\n                final_score=-1.0,\n                contradictions=["placeholder"],\n            )\n        ]\n\n    return candidates[: cfg.max_candidates], diagnostics\n',
    'submission': '"""Submission construction and strict validation."""\n\nfrom __future__ import annotations\n\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport pandas as pd\n\nfrom .chemistry import inchikey14_from_smiles, is_valid_smiles\nfrom .config import MAX_CANDIDATES\n\n\nREQUIRED_COLUMNS = ("molecule_id", "smiles")\n\n\ndef format_smiles_cell(smiles_list: Sequence[str], max_n: int = MAX_CANDIDATES) -> str:\n    cleaned = []\n    seen_ik = set()\n    for smi in smiles_list:\n        if smi is None:\n            continue\n        s = str(smi).strip()\n        if not s or ";" in s:\n            continue\n        if not is_valid_smiles(s):\n            continue\n        ik = inchikey14_from_smiles(s)\n        if ik is None or ik in seen_ik:\n            continue\n        seen_ik.add(ik)\n        cleaned.append(s)\n        if len(cleaned) >= max_n:\n            break\n    if not cleaned:\n        cleaned = ["CCO"]\n    return ";".join(cleaned)\n\n\ndef build_submission_frame(\n    predictions: Dict[str, Sequence[str]],\n    sample_submission: pd.DataFrame,\n) -> pd.DataFrame:\n    rows = []\n    for mid in sample_submission["molecule_id"].tolist():\n        smiles = predictions.get(mid, ["CCO"])\n        rows.append({"molecule_id": mid, "smiles": format_smiles_cell(smiles)})\n    return pd.DataFrame(rows, columns=list(REQUIRED_COLUMNS))\n\n\ndef validate_submission(\n    sub: pd.DataFrame,\n    sample_submission: pd.DataFrame,\n    test_molecule_ids: Optional[Iterable[str]] = None,\n) -> List[str]:\n    """Return list of error strings; empty means OK."""\n    errors: List[str] = []\n    if list(sub.columns) != list(REQUIRED_COLUMNS):\n        errors.append(f"columns must be {REQUIRED_COLUMNS}, got {list(sub.columns)}")\n        return errors\n\n    if sub.isnull().any().any():\n        errors.append("null values present")\n\n    sample_ids = list(sample_submission["molecule_id"])\n    sub_ids = list(sub["molecule_id"])\n    if sub_ids != sample_ids:\n        if set(sub_ids) != set(sample_ids):\n            missing = set(sample_ids) - set(sub_ids)\n            extra = set(sub_ids) - set(sample_ids)\n            errors.append(f"molecule_id set mismatch missing={len(missing)} extra={len(extra)}")\n        else:\n            errors.append("molecule_id order differs from sample_submission")\n\n    if sub["molecule_id"].duplicated().any():\n        errors.append("duplicate molecule_id rows")\n\n    if test_molecule_ids is not None:\n        test_set = set(test_molecule_ids)\n        if set(sub_ids) != test_set:\n            errors.append("submission ids != test molecule ids")\n\n    for i, row in sub.iterrows():\n        cell = row["smiles"]\n        if cell is None or str(cell).strip() == "":\n            errors.append(f"empty smiles at row {i}")\n            continue\n        parts = str(cell).split(";")\n        if len(parts) > MAX_CANDIDATES:\n            errors.append(f">{row[\'molecule_id\']}: >{MAX_CANDIDATES} candidates")\n        seen = set()\n        for smi in parts:\n            if ";" in smi and smi != str(cell):\n                errors.append(f"semicolon inside smiles token near {row[\'molecule_id\']}")\n            if not is_valid_smiles(smi):\n                errors.append(f"{row[\'molecule_id\']}: invalid SMILES {smi!r}")\n                break\n            ik = inchikey14_from_smiles(smi)\n            if ik is None:\n                errors.append(f"{row[\'molecule_id\']}: cannot make inchikey14 for {smi!r}")\n                break\n            if ik in seen:\n                errors.append(f"{row[\'molecule_id\']}: duplicate inchikey14 {ik}")\n                break\n            seen.add(ik)\n        if len(errors) > 50:\n            errors.append("too many errors; truncated")\n            break\n    return errors\n\n\ndef write_submission(sub: pd.DataFrame, path) -> None:\n    sub.to_csv(path, index=False)\n',
    'metrics': '"""Grouped validation metrics (MRR@25, Hit@k)."""\n\nfrom __future__ import annotations\n\nfrom typing import Dict, Iterable, List, Sequence\n\nimport numpy as np\n\n\ndef reciprocal_rank(pred_ik14: Sequence[str], true_ik14: str, k: int = 25) -> float:\n    if not true_ik14:\n        return 0.0\n    for i, ik in enumerate(pred_ik14[:k]):\n        if ik == true_ik14:\n            return 1.0 / (i + 1)\n    return 0.0\n\n\ndef hit_at_k(pred_ik14: Sequence[str], true_ik14: str, k: int) -> float:\n    return 1.0 if true_ik14 in list(pred_ik14)[:k] else 0.0\n\n\ndef summarize_metrics(\n    predictions: Dict[str, Sequence[str]],\n    truths: Dict[str, str],\n) -> Dict[str, float]:\n    """predictions: molecule_id -> ranked inchikey14 list; truths: molecule_id -> ik14."""\n    ids = sorted(set(predictions) & set(truths))\n    if not ids:\n        return {"n": 0}\n    mrr = []\n    hits = {k: [] for k in (1, 5, 10, 25)}\n    for mid in ids:\n        pred = list(predictions[mid])\n        true = truths[mid]\n        mrr.append(reciprocal_rank(pred, true, 25))\n        for k in hits:\n            hits[k].append(hit_at_k(pred, true, k))\n    out = {\n        "n": float(len(ids)),\n        "mrr@25": float(np.mean(mrr)),\n        "hit@1": float(np.mean(hits[1])),\n        "hit@5": float(np.mean(hits[5])),\n        "hit@10": float(np.mean(hits[10])),\n        "hit@25": float(np.mean(hits[25])),\n    }\n    return out\n',
}
(PKG/'casmi26'/'__init__.py').write_text('__version__="0.1.0"\n')
for name, src in SOURCES.items():
    (PKG/'casmi26'/f'{name}.py').write_text(src)
print('embedded casmi26 modules:', sorted(SOURCES))


In [ ]:
from casmi26.config import RunConfig, PREFERRED_LIBS
from casmi26.index import build_index_from_train
from casmi26.retrieve import aggregate_molecule_candidates
from casmi26.submission import build_submission_frame, validate_submission, write_submission
from tqdm.auto import tqdm

cfg = RunConfig()
print(json.dumps(cfg.to_dict(), indent=2))
t0=time.time()
index = build_index_from_train(
    DATA/'train.parquet', cfg, cache_path=ART/'spectrum_index.npz', libraries=None,
)
print(f'index size={len(index.neutral_mass)} build_elapsed={time.time()-t0:.1f}s')

test_df = pq.read_table(DATA/'test.parquet').to_pandas()
sample = pd.read_csv(DATA/'sample_submission.csv')
print('test rows', len(test_df), 'molecules', test_df.molecule_id.nunique())

predictions={}; evidence_rows=[]; stats={'fallback_molecules':0,'placeholder_molecules':0,'n_candidates_total':0}
t1=time.time()
for molecule_id, g in tqdm(list(test_df.groupby('molecule_id', sort=False)), desc='infer'):
    cands, diag = aggregate_molecule_candidates(molecule_id, g, index, cfg)
    predictions[molecule_id]=[c.smiles for c in cands]
    stats['fallback_molecules'] += int(bool(diag.get('fallback_used')))
    stats['placeholder_molecules'] += int(bool(diag.get('placeholder_used')))
    stats['n_candidates_total'] += len(cands)
    for rank,c in enumerate(cands,1):
        row=c.to_dict(); row.update({'molecule_id':molecule_id,'rank':rank,'neutral_mass':diag.get('neutral_mass')})
        evidence_rows.append(row)
stats['elapsed_sec']=time.time()-t1
stats['n_molecules']=len(predictions)
print('inference', stats)

sub = build_submission_frame(predictions, sample)
errors = validate_submission(sub, sample, test_molecule_ids=test_df.molecule_id.unique())
assert not errors, errors
write_submission(sub, OUT/'submission.csv')
pd.DataFrame(evidence_rows).to_parquet(OUT/'candidate_evidence.parquet', index=False)
cand_counts = sub['smiles'].apply(lambda s: len(str(s).split(';')))
summary={
  'n_rows': len(sub),
  'candidate_count_mean': float(cand_counts.mean()),
  'candidate_count_min': int(cand_counts.min()),
  'candidate_count_max': int(cand_counts.max()),
  'stats': stats,
  'index_size': int(len(index.neutral_mass)),
  'total_elapsed_sec': time.time()-t0,
}
print('=== RUN SUMMARY ===')
print(json.dumps(summary, indent=2))
(OUT/'run_summary.json').write_text(json.dumps(summary, indent=2))
print('Wrote', OUT/'submission.csv')
display(sub.head())
